# CACT Momentum Timing — DSL Backtest
**Signal:** `log(close_T / close_{T-w}) ≥ θ` → long MOC | **Params:** w=139, θ=0.005 (+0.50%) | **Period:** 2000–2026

In [1]:
import sys, warnings
sys.path.insert(0, r'c:\Personal\Business & Investments\Python codes')
sys.path.insert(0, r'c:\Personal\Business & Investments\Python codes\btest\src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sfera_db
from signum import Chart, Dashboard

from quantdsl_backtest.dsl.strategy import Strategy
from quantdsl_backtest.dsl.data_config import DataConfig
from quantdsl_backtest.dsl.universe import Universe
from quantdsl_backtest.dsl.factors import FieldFactor
from quantdsl_backtest.dsl.signals import MaskFromBoolean, GreaterEqual
from quantdsl_backtest.dsl.portfolio import TimingPortfolio
from quantdsl_backtest.dsl.execution import Execution, OrderPolicy, LatencyModel, PowerLawSlippageModel, VolumeParticipation
from quantdsl_backtest.dsl.costs import Costs, Commission, BorrowCost, FinancingCost, StaticFees
from quantdsl_backtest.dsl.backtest_config import BacktestConfig, RiskChecks, DrawdownPolicy, Reporting
from quantdsl_backtest.engine.analytics.types import StrategyAnalyticsConfig
from quantdsl_backtest.runners.single_asset import SingleAssetRunner

MOM_WINDOW = 139
THETA      = 0.005
START      = '2000-01-01'
END        = '2026-03-31'
BTEST_ROOT = r'c:\Personal\Business & Investments\Python codes\btest'

In [2]:
df = sfera_db.query("""
    SELECT trade_date AS date, open_price AS open, high_price AS high,
           low_price AS low, close_price AS close
    FROM bbgidx.index_total_return
    WHERE ticker = 'CACT' ORDER BY trade_date
""")
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index().loc[START:END]

mom  = np.log(df['close'] / df['close'].shift(MOM_WINDOW))
gate = (mom.shift(1) >= THETA).astype(float)
mask = mom.notna()

ret_cc  = np.log(df['close'] / df['close'].shift(1))
strat_r = (ret_cc * gate).fillna(0.0)
bh_r    = ret_cc.fillna(0.0)
strat_eq = np.exp(strat_r[mask].cumsum()); strat_eq = strat_eq / strat_eq.iloc[0] * 100
bh_eq    = np.exp(bh_r[mask].cumsum());    bh_eq    = bh_eq    / bh_eq.iloc[0]    * 100

df_aln = df[mask].copy(); df_aln['position'] = gate[mask].values

Dashboard([
    Chart(height=300, watermark='CACT').candlestick(df_aln).shade(df_aln, position_col='position'),
    Chart(height=110).baseline(pd.DataFrame({'time': df.index[mask], 'value': mom[mask].values}), base_value=THETA),
    Chart(height=150).line(pd.DataFrame({'time': df.index[mask], 'value': strat_eq.values}), label='Strategy')
                     .line(pd.DataFrame({'time': df.index[mask], 'value': bh_eq.values}),    label='B&H'),
]).show()

In [3]:
# ── DSL strategy + SingleAssetRunner ─────────────────────────────────────────
# ReturnFactor is not evaluated by SingleAssetRunner — use FieldFactor instead,
# passing the pre-computed momentum series (already in `mom`) via aux_series.
FACTOR_KEY = f'mom_{MOM_WINDOW}'

strategy = Strategy(
    name='cact_momentum_timing',
    data=DataConfig(source='sfera://bbgidx/index_total_return', calendar='XPAR',
                    frequency='1d', start=START, end=END),
    universe=Universe(name='CACT_TR', static_instruments=['CACT']),
    factors={FACTOR_KEY: FieldFactor(name=FACTOR_KEY, field=FACTOR_KEY)},
    signals={'entry_signal': MaskFromBoolean(
        name='entry_signal',
        expr=GreaterEqual(left=FACTOR_KEY, right=THETA),
    )},
    portfolio=TimingPortfolio(
        signal_name='entry_signal', instrument='CACT',
        rebalance_frequency='1d', rebalance_at='market_close',
        signal_delay_bars=1, target_leverage=1.0,
    ),
    execution=Execution(
        order_policy=OrderPolicy(default_order_type='MOC'), latency=LatencyModel(),
        slippage=PowerLawSlippageModel(base_bps=1.0, k=0.0),
        volume_limits=VolumeParticipation(max_participation=1.0),
    ),
    costs=Costs(
        commission=Commission(type='bps_notional', amount=1.0),
        borrow=BorrowCost(default_annual_rate=0.0),
        financing=FinancingCost(base_rate_curve='SOFR', spread_bps=0.0),
        fees=StaticFees(nav_fee_annual=0.0, perf_fee_fraction=0.0),
    ),
    backtest=BacktestConfig(
        engine='event_driven', cash_initial=1_000_000.0,
        risk_checks=RiskChecks(max_gross_leverage=1.0, drawdown=DrawdownPolicy(mode='none')),
        reporting=Reporting(output_dir='outputs/cact_momentum_timing', store_trades=True,
                            store_positions=True,
                            strategyAnalytics=StrategyAnalyticsConfig(
                                title=f'CACT Momentum  w={MOM_WINDOW}  theta={THETA}')),
    ),
)

res = SingleAssetRunner(strategy, btest_root=BTEST_ROOT).run(
    price_close=df['close'],
    aux_series={FACTOR_KEY: mom},   # pre-computed log-momentum series from cell 3
)

# ── Results ───────────────────────────────────────────────────────────────────
def stats(s, label):
    ann = s.mean()*252; vol = s.std()*np.sqrt(252)
    sh  = ann/vol; cagr = np.exp(ann)-1; total = np.exp(s.sum())-1
    mdd = (np.exp(s.cumsum()) / np.exp(s.cumsum()).cummax() - 1).min()
    print(f"{label:32s}  Sharpe={sh:.3f}  CAGR={cagr:.2%}  Total={total:.1%}  MaxDD={mdd:.1%}  InMkt={(s!=0).mean():.1%}")

stats(res['strat_ret'], f'Momentum w={MOM_WINDOW} theta={THETA}')
stats(res['daily_ret'], 'Buy & Hold')

idx   = res['strat_ret'].index
eq_s  = np.exp(res['strat_ret'].cumsum()); eq_s  = eq_s  / eq_s.iloc[0]  * 100
eq_bh = np.exp(res['daily_ret'].cumsum()); eq_bh = eq_bh / eq_bh.iloc[0] * 100

Chart(height=300, watermark='CACT Momentum').line(
    pd.DataFrame({'time': idx, 'value': eq_s.values}),  label='Strategy (net costs)'
).line(
    pd.DataFrame({'time': idx, 'value': eq_bh.values}), label='B&H'
).show()

Momentum w=139 theta=0.005        Sharpe=0.458  CAGR=5.91%  Total=361.1%  MaxDD=-22.3%  InMkt=65.5%
Buy & Hold                        Sharpe=0.191  CAGR=4.23%  Total=201.7%  MaxDD=-63.6%  InMkt=100.0%


In [6]:
# ── Results helper ────────────────────────────────────────────────────────────
def stats(s, label):
    ann = s.mean()*252; vol = s.std()*np.sqrt(252)
    sh  = ann/vol; cagr = np.exp(ann)-1; total = np.exp(s.sum())-1
    eq  = np.exp(s.cumsum())
    mdd = (eq / eq.cummax() - 1).min()
    mdd_date = (eq / eq.cummax() - 1).idxmin()
    print(f"{label:35s}  Sharpe={sh:.3f}  CAGR={cagr:.2%}  Total={total:.1%}  MaxDD={mdd:.1%} ({mdd_date.date()})  InMkt={(s!=0).mean():.1%}")

def run_lev(lev, fin_spread_bps):
    """Build + run a DSL strategy at given leverage with IBKR EUR financing."""
    s = Strategy(
        name=f'cact_mom_{lev}x',
        data=DataConfig(source='sfera://bbgidx/index_total_return', calendar='XPAR',
                        frequency='1d', start=START, end=END),
        universe=Universe(name='CACT_TR', static_instruments=['CACT']),
        factors={FACTOR_KEY: FieldFactor(name=FACTOR_KEY, field=FACTOR_KEY)},
        signals={'entry_signal': MaskFromBoolean(
            name='entry_signal',
            expr=GreaterEqual(left=FACTOR_KEY, right=THETA),
        )},
        portfolio=TimingPortfolio(
            signal_name='entry_signal', instrument='CACT',
            rebalance_frequency='1d', rebalance_at='market_close',
            signal_delay_bars=1, target_leverage=float(lev),
        ),
        execution=Execution(
            order_policy=OrderPolicy(default_order_type='MOC'), latency=LatencyModel(),
            slippage=PowerLawSlippageModel(base_bps=1.0, k=0.0),
            volume_limits=VolumeParticipation(max_participation=1.0),
        ),
        costs=Costs(
            commission=Commission(type='bps_notional', amount=1.0),
            borrow=BorrowCost(default_annual_rate=0.0),
            # IBKR EUR Pro tier: €STR + 1.50%  (rate_csv_path auto-resolved via registry)
            financing=FinancingCost(base_rate_curve='ESTR', spread_bps=fin_spread_bps),
            fees=StaticFees(nav_fee_annual=0.0, perf_fee_fraction=0.0),
        ),
        backtest=BacktestConfig(
            engine='event_driven', cash_initial=1_000_000.0,
            risk_checks=RiskChecks(max_gross_leverage=float(lev), drawdown=DrawdownPolicy(mode='none')),
            reporting=Reporting(output_dir=f'outputs/cact_momentum_{lev}x', store_trades=False,
                                store_positions=False,
                                strategyAnalytics=StrategyAnalyticsConfig(
                                    title=f'CACT Momentum {lev}x  w={MOM_WINDOW}  theta={THETA}')),
        ),
    )
    return SingleAssetRunner(s, btest_root=BTEST_ROOT).run(
        price_close=df['close'],
        aux_series={FACTOR_KEY: mom},
    )

IBKR_SPREAD_BPS = 150   # €STR + 1.50%  (IBKR Pro EUR, balance < €90k tier)

res_1x = run_lev(1, 0)
res_2x = run_lev(2, IBKR_SPREAD_BPS)
res_3x = run_lev(3, IBKR_SPREAD_BPS)

print(f"Financing: €STR + {IBKR_SPREAD_BPS/100:.2f}% (IBKR Pro EUR)")
print()
stats(res_1x['strat_ret'], '1x  (no leverage)')
stats(res_2x['strat_ret'], '2x  (€STR + 1.50%)')
stats(res_3x['strat_ret'], '3x  (€STR + 1.50%)')
print()
stats(res_1x['daily_ret'], 'B&H (1x)')

# ── Cumulative costs & financing breakdown ────────────────────────────────────
# commission = sum of (1bp commission + 1bp slippage) per trade × leverage × notional
#              e.g. 1x: 173 trades × 2bps = 3.46%  |  2x: same × 2 = 6.92%
# financing  = cumulative daily interest on borrowed portion (lev-1) × position
#              e.g. 2x borrows 1× notional at ~€STR+1.5%; 3x borrows 2× notional
print()
for r, lbl in [(res_1x,'1x'), (res_2x,'2x'), (res_3x,'3x')]:
    comm = r['cost_ret'].sum()*100
    fin  = r['financing_ret'].sum()*100
    n_tr = int((r['position'].diff().abs() > 0).sum())
    print(f"{lbl}  trades={n_tr}  commission={comm:.2f}%  financing={fin:.2f}%  total_drag={comm+fin:.2f}%")

# ── Build equity + drawdown series ───────────────────────────────────────────
idx      = res_1x['strat_ret'].index
eq_data  = {}
dd_data  = {}

for r, lbl in [(res_1x,'1x'), (res_2x,'2x'), (res_3x,'3x'), (res_1x,'BH')]:
    ret  = r['daily_ret'] if lbl == 'BH' else r['strat_ret']
    eq   = np.exp(ret.reindex(idx).fillna(0).cumsum())
    eq   = eq / eq.iloc[0] * 100
    dd   = (eq / eq.cummax() - 1) * 100          # drawdown in %
    eq_data[lbl] = pd.DataFrame({'time': idx, 'value': eq.values})
    dd_data[lbl] = pd.DataFrame({'time': idx, 'value': dd.values})

# Find max-DD date for 3x to mark it
mdd_3x_idx  = dd_data['3x']['value'].idxmin()
mdd_3x_date = dd_data['3x']['time'].iloc[mdd_3x_idx]
mdd_3x_val  = dd_data['3x']['value'].min()
print(f"\n3x MaxDD trough: {pd.Timestamp(mdd_3x_date).date()}  ({mdd_3x_val:.1f}%)")

# ── Charts ────────────────────────────────────────────────────────────────────
Dashboard([
    Chart(height=300, watermark='CACT Momentum — Leverage comparison')
        .line(eq_data['1x'],  label='1x')
        .line(eq_data['2x'],  label='2x')
        .line(eq_data['3x'],  label='3x')
        .line(eq_data['BH'],  label='B&H'),
    Chart(height=160, watermark='Drawdown %')
        .line(dd_data['1x'],  label='1x')
        .line(dd_data['2x'],  label='2x')
        .line(dd_data['3x'],  label='3x'),
]).show()

Financing: €STR + 1.50% (IBKR Pro EUR)

1x  (no leverage)                    Sharpe=0.458  CAGR=5.91%  Total=361.1%  MaxDD=-22.3% (2012-05-15)  InMkt=65.5%
2x  (€STR + 1.50%)                   Sharpe=0.390  CAGR=10.27%  Total=1249.8%  MaxDD=-40.9% (2012-05-15)  InMkt=65.5%
3x  (€STR + 1.50%)                   Sharpe=0.367  CAGR=14.80%  Total=3850.9%  MaxDD=-55.1% (2012-05-15)  InMkt=65.5%

B&H (1x)                             Sharpe=0.191  CAGR=4.23%  Total=201.7%  MaxDD=-63.6% (2003-03-12)  InMkt=100.0%

1x  trades=173  commission=3.46%  financing=0.00%  total_drag=3.46%
2x  trades=173  commission=6.92%  financing=45.46%  total_drag=52.38%
3x  trades=173  commission=10.38%  financing=90.91%  total_drag=101.29%

3x MaxDD trough: 2012-05-15  (-55.1%)
